## 의사결정나무
- 질문에 따라 판단
- 젤 위 루트 노드 -> 분할 -> 분할....
- 깊이가 짚어질 때 과적합이 발생할 위험이 크다.
- 분류 / 회귀 둘 다 가능

In [2]:
# 실습 폴더 불러오기
from pathlib import Path    
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('ml_실습데이터')
if not DATA_DIR.exists():
    DATA_DIR = Path('ml_실습데이터')
print(f'데이터 폴더: {DATA_DIR.resolve()}')

데이터 폴더: C:\Users\Administrator\bigdata2026\data_analysis\ml_실습데이터


### 라이브러리 불러오기

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
# export_text : 나무 구조를 글자로 출력
from sklearn.tree import DecisionTreeClassifier, export_text  # 의사결정나무 분류 
from sklearn.impute import SimpleImputer # 결측치를 특정 규칙에 따라 채워주는 전처리 도구
from sklearn.pipeline import make_pipeline # 여러 전처리 단계와 모델을 하나의 객체로 순서대로 연결

### 데이터 불러오기

In [4]:
df = pd.read_csv(DATA_DIR/'telecom_churn.csv')
df.head()

,usage_minutes,complaints,contract_months,monthly_fee,contract_type,region,churn
0,321.0,2,12,63.6,one-year,Gyeonggi,1
1,80.0,1,6,62.2,two-year,Gyeonggi,1
2,251.0,1,31,71.6,month-to-month,Other,0
3,158.0,0,31,58.8,two-year,Seoul,0
4,356.0,1,32,58.9,one-year,Gyeonggi,0


### 피처(입력), 타겟(정답) 데이터 나누기

In [5]:
# usage_minutes : 고객의 (월간) 통화/이용 시간 (분) -> 이용량이 급격히 줄어든 고객은 이미 다른 서비스로 떠났다.
# complaints : 고객이 접수한 불만/민원 건수 -> 서비스에 불만족해서 이탈할 확률이 높다고 판단
# contract_months : 계약 유지 기간(개월 수) -> 계약 기간이 짧을 수록 이탈 위험이 크다고 통신업계의 관찰
# monthly_fee : 월 요금 (청구액) -> 요금이 부담스러울수록 이탈 가능성이 있다. 
# churn : 타겟(정답), 이탈한다(1) / 안한다(0)
features = ['usage_minutes', 'complaints', 'contract_months', 'monthly_fee'] # 입력 컬럼
X = df[features]
y = df['churn']

X.shape, y.shape

((420, 4), (420,))

### 훈련 / 검증 데이터 나누기

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_valid.shape, y_train.shape, y_valid.shape

((315, 4), (105, 4), (315,), (105,))

### 의사결정나무 분류 모델 학습시키기

In [7]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)  # 학습

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [8]:
pred = model.predict(X_valid)  # 예측값

In [9]:
from sklearn.metrics import accuracy_score # 정확도

accuracy_score(y_valid, pred) # 정답과 예측을 비교해서 얼마나 정확한가?

0.5333333333333333

### 나무 깊이를 바꿔가며 훈련/검증 점수를 비교 -> 과적합 관찰

In [10]:
# 깊이를 2, 4, 제한없음(None) 세 가지로 바꿔가며 성능 비교
for depth in [2, 4, None]:
    model = make_pipeline(
        SimpleImputer(strategy='median'), # 결측치 패턴을 중앙값으로 채운다.
        DecisionTreeClassifier(max_depth=depth, random_state=42)
    )
    model.fit(X_train, y_train) # 학습
    pred = model.predict(X_valid)  # 예측
    print(f'{depth} -> {accuracy_score(y_valid, pred):.4f}') # 정확도 확인(소수 4째자리까지 확인)

2 -> 0.6952
4 -> 0.6476
None -> 0.5333


- 깊이가 없음(끝까지 내려가는 것)과 깊이 2이 성능이 서로 비슷하다면
- 깊이가 얕은 모델을 우선한다. 

### 추가) 깊이가 2인 얕은 나무모델을 직접 뜯어보기

In [11]:
shallow = make_pipeline(
    SimpleImputer(strategy='median'),
    DecisionTreeClassifier(max_depth=2, random_state=42)
)
shallow.fit(X_train, y_train)

# named_steps: 파이프라인 안에서 이름으로 특정 단계(모델)를 꺼내오는 방법
tree_model = shallow.named_steps['decisiontreeclassifier']

print(export_text(tree_model, feature_names=features))

|--- complaints <= 1.50
|   |--- contract_months <= 12.50
|   |   |--- class: 0
|   |--- contract_months >  12.50
|   |   |--- class: 0
|--- complaints >  1.50
|   |--- monthly_fee <= 32.75
|   |   |--- class: 0
|   |--- monthly_fee >  32.75
|   |   |--- class: 1



In [12]:
shallow

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('simpleimputer', ...), ('decisiontreeclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness desp

In [13]:
shallow.named_steps

{'simpleimputer': SimpleImputer(strategy='median'),
 'decisiontreeclassifier': DecisionTreeClassifier(max_depth=2, random_state=42)}

## 여러 모델의 의견을 모으는 앙상블
- 랜덤포레스트 --> 데이터를 무작위로 복원추출(부트스트랩), 
            각 분할마다 특성도 무작위로 일부만 사용해서 서로 다른 나무 여러 개를 만든 뒤 평균/다수결로 예측
            단일 나무보다 과적합에 강하지만, 개별 나무 하나하를 사람이 읽고 설명하지는 어려워진다라는
            트레이드오프가 있다.
            '여러 사람에게 물어보고 다수결로 정한다'라는 비유
            분류 / 회귀 둘 다 가능
            분산 처리(배깅) 다수결(보팅)

### 라이브러리 불러오기

In [14]:
from sklearn.ensemble import RandomForestClassifier # 여러 개의 의사결정나무를 모아 만드는 앙상블 모델
from sklearn.metrics import f1_score  # 정밀도와 재현율의 조화평균(불균형 데이터에 적합하다)

In [15]:
df = pd.read_csv(DATA_DIR/'telecom_churn.csv')
features

['usage_minutes', 'complaints', 'contract_months', 'monthly_fee']

In [16]:
# 입력(피처), 정답(타겟) --> 훈련/검증 나누기
X = df[features]
y = df['churn']
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
X_train.shape, X_valid.shape, y_train.shape, y_valid.shape

((315, 4), (105, 4), (315,), (105,))

In [17]:
model = RandomForestClassifier(
    n_estimators=200,  # 나무가 200그루
    max_depth=5,
    random_state=42
)

# 학습
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [18]:
# 예측
pred = model.predict(X_valid)

# 평가 - 정확도
print(f'랜덤포레스트 분류의 정확도: {accuracy_score(y_valid, pred):.4f}')

랜덤포레스트 분류의 정확도: 0.6476


### 의사결정나무와 랜덤포레스트 비교

In [19]:
models = {
    'tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'forest': RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42)
}
models.keys()

dict_keys(['tree', 'forest'])

In [20]:
models.values()

dict_values([DecisionTreeClassifier(max_depth=4, random_state=42), RandomForestClassifier(max_depth=4, n_estimators=200, random_state=42)])

In [21]:
models.items() #  튜플로 쌍을 묶는다

dict_items([('tree', DecisionTreeClassifier(max_depth=4, random_state=42)), ('forest', RandomForestClassifier(max_depth=4, n_estimators=200, random_state=42))])

In [22]:
fitted = {}
for name, estimator in models.items():
    pipe = make_pipeline(SimpleImputer(strategy='median'), estimator)
    pipe.fit(X_train, y_train)  # 학습
    fitted[name] = pipe  # 나중에 다시 꺼내쓰기 위해서 딕셔너리에 저장
    pred = pipe.predict(X_valid) # 예측
    print(f'{name} F1 score: {f1_score(y_valid, pred):.4f}')  # f1 score(조화평균) 확인
    print(f'{name} 정확도: {accuracy_score(y_valid, pred):.4f}')  # 정확도 확인

tree F1 score: 0.4127
tree 정확도: 0.6476
forest F1 score: 0.4000
forest 정확도: 0.6571


In [23]:
fitted

{'tree': Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                 ('decisiontreeclassifier',
                  DecisionTreeClassifier(max_depth=4, random_state=42))]),
 'forest': Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                 ('randomforestclassifier',
                  RandomForestClassifier(max_depth=4, n_estimators=200,
                                         random_state=42))])}

### 랜덤포레스트가 어떤 특성을 중요하게 봤는지 확인

In [24]:
forest = fitted['forest'].named_steps['randomforestclassifier']

In [25]:
importance = forest.feature_importances_  #  특성 중요도
importance

array([0.20330039, 0.3036125 , 0.26338513, 0.22970198])

In [26]:
features

['usage_minutes', 'complaints', 'contract_months', 'monthly_fee']

In [30]:
df2 = pd.DataFrame(
    data=importance,
    index=features,
    columns=['특성 중요도']
)
df2.sort_values('특성 중요도', ascending=False)

,특성 중요도
complaints,0.303612
contract_months,0.263385
monthly_fee,0.229702
usage_minutes,0.203300


In [ ]:
df_importance = pd.DataFrame({
    '특성(컬럼)': features,
    '중요도': importance,
})
df_importance

,특성(컬럼),중요도
0,usage_minutes,0.203300
1,complaints,0.303612
2,contract_months,0.263385
3,monthly_fee,0.229702


In [35]:
# 나무 수를 50으로 바꾸고, 정확도, f1-score구하기
model_50 = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    random_state=42
)
model_50.fit(X_train, y_train)
pred_50 = model_50.predict(X_valid)
acc_50 = accuracy_score(y_valid, pred_50)
print(f'정확도: {acc_50:.4f}')

f1_50 = f1_score(y_valid, pred_50)
print(f'f1 스코어: {f1_50:.4f}')

df_50 = pd.DataFrame(data=model_50.feature_importances_,
                     index=features,
                     columns=['나무 50개일 때 특성 중요도'])
df_50.sort_values(by='나무 50개일 때 특성 중요도', ascending=False)

정확도: 0.6476
f1 스코어: 0.4127


,나무 50개일 때 특성 중요도
monthly_fee,0.309272
contract_months,0.238881
complaints,0.236786
usage_minutes,0.215061
